# Couche Gold — indicateurs prêts pour le dashboard

Ce notebook produit cinq tables agrégées à partir de Silver pour le classement géographique, les séries quotidiennes et mensuelles, le profil jour/heure et les KPI. Il les écrit dans la couche Gold du data lake puis les exporte vers PostgreSQL. Les enregistrements signalés `aberrant_q999` restent conservés dans Silver mais sont exclus de ces agrégats. Les comptages décrivent les mesures disponibles, pas la capacité des rues ni leur taux de saturation.

## 1. Connexion et contrôles de la source

Les chemins, la session Spark et le format de stockage suivent les conventions du projet. Exécuter d'abord `work/02_silver.ipynb` pour produire Silver. Le notebook accepte Delta ou le format de repli activé par `commun.spark_utils`.

In [ ]:
from pyspark.sql import Window, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.storagelevel import StorageLevel
from commun.spark_utils import LAKE, PG_PROPS, PG_URL, creer_session, format_tables, vers_postgres

spark = creer_session("S03_Gold", lac=True, delta=True, postgres=True, master="local[2]")
spark.conf.set("spark.sql.session.timeZone", "Europe/Paris")
FMT = format_tables(spark)
SILVER_PATH = f"{LAKE}/silver/comptage_multimodal_paris"
GOLD_PATH = f"{LAKE}/gold"

silver = spark.read.format(FMT).load(SILVER_PATH)
colonnes_requises = {
    "identifiant_site", "nom_du_site", "latitude", "longitude", "geo_connue",
    "mode_deplacement", "nombre_vehicules", "date_heure",
    "annee", "mois", "jour_semaine", "heure", "aberrant_q999",
}
manquantes = colonnes_requises - set(silver.columns)
if manquantes:
    raise ValueError(f"Colonnes Silver manquantes : {sorted(manquantes)}")

nb_silver = silver.count()
nb_aberrantes = silver.filter(F.col("aberrant_q999") == F.lit(True)).count()
nb_flag_absent = silver.filter(F.col("aberrant_q999").isNull()).count()
if nb_flag_absent:
    raise ValueError(f"{nb_flag_absent:,} lignes Silver n'ont pas de valeur aberrant_q999")

propre = silver.filter(F.col("aberrant_q999") == F.lit(False)).persist(StorageLevel.MEMORY_AND_DISK)
nb_propre = propre.count()
print(f"Format : {FMT}")
print(f"Silver : {nb_silver:,} lignes")
print(f"Lignes signalées et exclues des agrégats Gold : {nb_aberrantes:,}")
print(f"Lignes retenues pour Gold : {nb_propre:,}")
if nb_propre == 0:
    raise ValueError("Aucune ligne non aberrante à agréger dans Silver.")

## 2. Construire les cinq agrégats

Les données sources peuvent contenir plusieurs trajectoires pour un site, un mode et une même heure. On additionne d'abord ces trajectoires dans une observation **site × mode × heure mesurée**; les moyennes portent ensuite sur ces heures effectivement observées, sans inventer les heures sans mesure. Le classement géographique est calculé par site, mode, année et mois, puis classé dans chaque période et mode. La série quotidienne répond à l'évolution par date, la série mensuelle aux tendances de fond, et la carte de chaleur compare les profils par jour de semaine et heure.

In [ ]:
heure = F.date_trunc("hour", F.col("date_heure"))

# Une ligne par site, mode et heure réellement observée, tous axes/trajectoires confondus.
site_heure = (propre
    .withColumn("heure_mesuree", heure)
    .groupBy("identifiant_site", "mode_deplacement", "heure_mesuree")
    .agg(F.sum("nombre_vehicules").alias("total_heure"))
    .withColumn("annee", F.year("heure_mesuree"))
    .withColumn("mois", F.month("heure_mesuree"))
    .withColumn("jour_semaine", F.dayofweek("heure_mesuree"))
    .withColumn("heure", F.hour("heure_mesuree"))
    .persist(StorageLevel.MEMORY_AND_DISK))

# Le référentiel évite de dédoubler un site si ses libellés ou coordonnées varient dans la source.
coordonnees_site = (propre
    .filter(F.col("latitude").between(-90, 90) & F.col("longitude").between(-180, 180))
    .groupBy("identifiant_site")
    .agg(F.min(F.struct("latitude", "longitude")).alias("coordonnees_ref")))
site_reference = (propre.groupBy("identifiant_site")
    .agg(F.min("nom_du_site").alias("nom_du_site"))
    .join(coordonnees_site, "identifiant_site", "left")
    .select("identifiant_site", "nom_du_site",
            F.col("coordonnees_ref.latitude").alias("latitude"),
            F.col("coordonnees_ref.longitude").alias("longitude")))
fenetre_classement = Window.partitionBy("annee", "mois", "mode_deplacement").orderBy(F.desc("total_usagers"))
gold_sites_classement = (site_heure
    .groupBy("identifiant_site", "mode_deplacement", "annee", "mois")
    .agg(F.sum("total_heure").cast("long").alias("total_usagers"),
         F.round(F.avg("total_heure"), 2).alias("moyenne_par_heure"),
         F.count("*").cast("long").alias("nb_heures_mesurees"))
    .join(site_reference, "identifiant_site", "left")
    .withColumn("geo_connue", F.col("latitude").between(-90, 90) & F.col("longitude").between(-180, 180))
    .withColumn("rang", F.dense_rank().over(fenetre_classement))
    .select("rang", "identifiant_site", "nom_du_site", "latitude", "longitude", "geo_connue",
            "mode_deplacement", "annee", "mois", "total_usagers",
            "moyenne_par_heure", "nb_heures_mesurees"))

# Séries mensuelles : moyennes sur les observations site-mode-heure présentes.
gold_serie_mensuelle = (site_heure
    .groupBy("annee", "mois", "mode_deplacement")
    .agg(F.sum("total_heure").cast("long").alias("total_usagers"),
         F.countDistinct("identifiant_site").cast("long").alias("nb_sites_actifs"),
         F.round(F.avg("total_heure"), 2).alias("moyenne_par_heure"),
         F.count("*").cast("long").alias("nb_sites_heures_mesurees")))

# Série quotidienne : un total par date et mode, à partir des heures site-mode observées.
gold_serie_quotidienne = (site_heure
    .withColumn("jour", F.to_date("heure_mesuree"))
    .groupBy("jour", "mode_deplacement")
    .agg(F.sum("total_heure").cast("long").alias("total_usagers"),
         F.countDistinct("identifiant_site").cast("long").alias("nb_sites_actifs"),
         F.round(F.avg("total_heure"), 2).alias("moyenne_par_heure"),
         F.count("*").cast("long").alias("nb_sites_heures_mesurees")))

# Profil jour/heure : chaque moyenne porte sur les heures site-mode effectivement observées.
gold_heatmap_heure_jour = (site_heure
    .groupBy("mode_deplacement", "jour_semaine", "heure")
    .agg(F.round(F.avg("total_heure"), 2).alias("moyenne_par_heure"),
         F.count("*").cast("long").alias("nb_sites_heures_mesurees"),
         F.countDistinct("identifiant_site").cast("long").alias("nb_sites_actifs")))

## 3. KPI de synthèse

La table KPI est au format long pour faciliter l'affichage dans un tableau de bord. Elle contient cinq indicateurs globaux. L'heure de pointe est celle dont le total moyen mesuré par site-heure (tous modes réunis) est le plus élevé; les ex æquo sont départagés par l'heure la plus tôt.

In [ ]:
total_usagers = site_heure.agg(F.sum("total_heure").alias("total")).first()["total"]
nb_sites_actifs = site_heure.select("identifiant_site").distinct().count()
if total_usagers is None or total_usagers <= 0:
    raise ValueError("Le total des comptages Silver non aberrants doit être strictement positif.")

totaux_modes = (site_heure.groupBy("mode_deplacement")
    .agg(F.sum("total_heure").alias("total_mode")))
mode_principal = (totaux_modes.orderBy(F.desc("total_mode"), F.asc("mode_deplacement"))
    .first())
part_mode_principal = 100.0 * mode_principal["total_mode"] / total_usagers

totaux_site_heure = (propre
    .withColumn("heure_mesuree", heure)
    .groupBy("identifiant_site", "heure_mesuree")
    .agg(F.sum("nombre_vehicules").alias("total_site_heure")))
heure_pointe = (totaux_site_heure
    .withColumn("heure", F.hour("heure_mesuree"))
    .groupBy("heure")
    .agg(F.avg("total_site_heure").alias("moyenne_site_heure"))
    .orderBy(F.desc("moyenne_site_heure"), F.asc("heure"))
    .first())

schema_kpi = StructType([
    StructField("kpi_id", StringType(), False),
    StructField("libelle", StringType(), False),
    StructField("valeur_numerique", DoubleType(), True),
    StructField("valeur_texte", StringType(), True),
    StructField("unite", StringType(), False),
    StructField("definition", StringType(), False),
])
kpi_rows = [
    ("total_usagers", "Total des véhicules comptés", float(total_usagers), None, "véhicules",
     "Somme des comptages non aberrants sur toute la période."),
    ("sites_actifs", "Nombre de sites actifs", float(nb_sites_actifs), None, "sites",
     "Nombre de sites ayant au moins une observation non aberrante."),
    ("mode_principal", "Mode avec le plus grand volume", None, mode_principal["mode_deplacement"], "mode",
     "Mode dont la somme des véhicules comptés est la plus élevée."),
    ("part_mode_principal_pct", "Part du mode principal", float(part_mode_principal), None, "%",
     "Part du mode principal dans le total des comptages non aberrants."),
    ("heure_pointe_moyenne", "Heure de pointe moyenne", float(heure_pointe["heure"]), None, "heure (0-23)",
     "Heure avec le plus grand total moyen par site-heure, tous modes réunis."),
]
gold_kpi = spark.createDataFrame(kpi_rows, schema_kpi)

## 4. Contrôles, écriture et relecture dans Gold

Les totaux du classement, des séries quotidiennes et mensuelles doivent correspondre à la somme Silver conservée après exclusion des lignes signalées. Les clés ne doivent pas être nulles et le grain doit être unique. Chaque table est écrite et relue dans `s3a://datalake/gold/<nom>`. L'export PostgreSQL est une étape distincte.

In [ ]:
def somme_colonne(table, colonne):
    resultat = table.agg(F.sum(colonne).alias("total")).first()["total"]
    return int(resultat or 0)

if somme_colonne(gold_sites_classement, "total_usagers") != int(total_usagers):
    raise ValueError("Le total du classement par site ne correspond pas au total de Silver non aberrante.")
if somme_colonne(gold_serie_mensuelle, "total_usagers") != int(total_usagers):
    raise ValueError("Le total des séries mensuelles ne correspond pas au total de Silver non aberrante.")
if somme_colonne(gold_serie_quotidienne, "total_usagers") != int(total_usagers):
    raise ValueError("Le total des séries quotidiennes ne correspond pas au total de Silver non aberrante.")

cles_obligatoires = {
    "gold_sites_classement": (gold_sites_classement, ["identifiant_site", "mode_deplacement", "annee", "mois", "rang"]),
    "gold_serie_mensuelle": (gold_serie_mensuelle, ["annee", "mois", "mode_deplacement"]),
    "gold_serie_quotidienne": (gold_serie_quotidienne, ["jour", "mode_deplacement"]),
    "gold_heatmap_heure_jour": (gold_heatmap_heure_jour, ["mode_deplacement", "jour_semaine", "heure"]),
    "gold_kpi": (gold_kpi, ["kpi_id", "libelle"]),
}
for nom, (table, cles) in cles_obligatoires.items():
    condition = None
    for colonne in cles:
        test = F.col(colonne).isNull()
        condition = test if condition is None else (condition | test)
    nuls = table.filter(condition).count()
    if nuls:
        raise ValueError(f"{nom} contient {nuls:,} lignes avec une clé obligatoire nulle.")

cles_uniques = {
    "gold_sites_classement": (gold_sites_classement, ["identifiant_site", "mode_deplacement", "annee", "mois"]),
    "gold_serie_mensuelle": (gold_serie_mensuelle, ["annee", "mois", "mode_deplacement"]),
    "gold_serie_quotidienne": (gold_serie_quotidienne, ["jour", "mode_deplacement"]),
    "gold_heatmap_heure_jour": (gold_heatmap_heure_jour, ["mode_deplacement", "jour_semaine", "heure"]),
    "gold_kpi": (gold_kpi, ["kpi_id"]),
}
for nom, (table, cles) in cles_uniques.items():
    doublons = table.groupBy(*cles).count().filter(F.col("count") > 1).limit(1).count()
    if doublons:
        raise ValueError(f"{nom} contient plusieurs lignes pour une même clé de grain.")

tables_gold = {
    "gold_sites_classement": gold_sites_classement,
    "gold_serie_mensuelle": gold_serie_mensuelle,
    "gold_serie_quotidienne": gold_serie_quotidienne,
    "gold_heatmap_heure_jour": gold_heatmap_heure_jour,
    "gold_kpi": gold_kpi,
}
for nom, table in tables_gold.items():
    chemin = f"{GOLD_PATH}/{nom}"
    ecriture = table.write.format(FMT).mode("overwrite").option("overwriteSchema", "true")
    if "annee" in table.columns:
        ecriture = ecriture.partitionBy("annee")
    ecriture.save(chemin)

    relue = spark.read.format(FMT).load(chemin)
    nb_attendu = table.count()
    nb_relu = relue.count()
    if nb_relu != nb_attendu:
        raise RuntimeError(f"{nom}: {nb_relu:,} lignes relues, {nb_attendu:,} attendues")
    print(f"{nom}: {nb_relu:,} lignes vérifiées dans le data lake — {chemin}")

print(f"Total Gold vérifié : {int(total_usagers):,} véhicules; total Silver non aberrant : {int(total_usagers):,}.")

## 5. Exporter les tables Gold vers PostgreSQL

Les cinq tables agrégées sont écrites dans le schéma `public`, sous leurs noms `gold_*`. Le mode `overwrite` remplace uniquement ces tables lors d'une nouvelle exécution. Chaque export est relu depuis PostgreSQL et son nombre de lignes est comparé à la table Spark.

In [ ]:
if set(tables_gold) != {
    "gold_sites_classement",
    "gold_serie_mensuelle",
    "gold_serie_quotidienne",
    "gold_heatmap_heure_jour",
    "gold_kpi",
}:
    raise ValueError(f"Ensemble de tables Gold inattendu : {sorted(tables_gold)}")

for nom, table in tables_gold.items():
    nb_attendu = table.count()
    vers_postgres(table, nom, mode="overwrite")
    relue_postgres = spark.read.jdbc(PG_URL, nom, properties=PG_PROPS)
    nb_postgres = relue_postgres.count()
    if nb_postgres != nb_attendu:
        raise RuntimeError(
            f"{nom}: {nb_postgres:,} lignes relues dans PostgreSQL, {nb_attendu:,} attendues"
        )
    print(f"public.{nom}: {nb_postgres:,} lignes vérifiées dans PostgreSQL ({PG_URL})")

## 6. Inspection des tables produites

Vérifier les premières lignes et les nombres de lignes de chaque table. `geo_connue = false` identifie les lieux à exclure d'une carte, sans les exclure des agrégats temporels.

In [ ]:
for nom, table in tables_gold.items():
    print(f"\n--- {nom} : {table.count():,} lignes ---")
    table.show(10, truncate=False)

print("Sites sans coordonnées valides :",
      gold_sites_classement.filter(~F.coalesce(F.col("geo_connue"), F.lit(False)))
      .select("identifiant_site").distinct().count())
gold_kpi.orderBy("kpi_id").show(truncate=False)
site_heure.unpersist()
propre.unpersist()